# Mining Operations Intelligence: Database Setup and Data Quality Audit

## Project Overview

This project audits a synthetic mining operations database obtained from GitHub. The database contains daily production, operating costs, equipment performance, safety incidents, and mine site information.

## Project Objectives

- Load and validate the SQLite database.
- Inspect database tables, schemas, constraints, and volume.
- Identify missing values and duplicate records.
- Validate key uniqueness, site references, and join integrity.
- Check cost consistency and unit-cost definitions.
- Test value ranges and logical rules across tables.
- Assess data quality before performing exploratory analysis.

## Tools and Technologies

- Python
- SQLite
- Pandas
- Google Colab

## Data Source

GitHub: FlavioColtellacci/Mining-Operations-Dashboard

The source project describes the data as synthetic. The findings are therefore used for analytical demonstration and are not presented as verified results from an actual mining company.

## Workflow

Database Setup → Schema Inspection → Data Profiling → Data Quality Audit → Cross-Table Validation → Audit Conclusion

The audit is followed by `02_Mining_Operations_EDA.ipynb`, which uses the same `mining_operations.db`.

## Notebook Structure

| Section | Topic |
|---|---|
| 1 | Environment and database connection |
| 2 | Database structure (tables, rows, schema, constraints, preview) |
| 3 | Date and site coverage |
| 4 | Missing values |
| 5 | Duplicate rows and key uniqueness |
| 6 | Production and cost reconciliation |
| 7 | Cost consistency |
| 8 | Cost per unit and zero-production records |
| 9 | Value range and logical consistency checks |
| 10 | Audit summary and conclusion |

> **Principle:** No original database values are modified, removed, or imputed during this audit.

## 1. Environment and Database Connection

In [1]:
import os
import sqlite3
from pathlib import Path

import pandas as pd
from google.colab import drive

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# Mount Google Drive (safe to re-run)
drive.mount('/content/drive')

print("Libraries imported successfully.")
print("Pandas version:", pd.__version__)

Mounted at /content/drive
Libraries imported successfully.
Pandas version: 3.0.2


In [2]:
# Locate the database file in Google Drive
db_candidates = [
    Path(root) / file
    for root, dirs, files in os.walk('/content/drive/MyDrive')
    for file in files
    if file.lower() == 'mining_operations.db'
]

print("Number of database files found:", len(db_candidates))
for path in db_candidates:
    print(path)

if not db_candidates:
    raise ValueError("Database file not found. Please place mining_operations.db in Google Drive.")

if len(db_candidates) > 1:
    print("\nWarning: more than one file found. The first one is used.")

DB_PATH = db_candidates[0]
print("\nFile size:", f"{DB_PATH.stat().st_size:,} bytes")

# Check that the file is a valid SQLite database
with open(DB_PATH, "rb") as file:
    file_signature = file.read(16)

if file_signature != b"SQLite format 3\x00":
    raise ValueError("Invalid SQLite database. Please check the file.")

# Connect and run SQLite's own integrity check
conn = sqlite3.connect(DB_PATH)
integrity = pd.read_sql_query("PRAGMA integrity_check", conn).iloc[0, 0]

print("\nDatabase validation successful.")
print("SQLite version:", sqlite3.sqlite_version)
print("Integrity check:", integrity)

Number of database files found: 1
/content/drive/MyDrive/Mining-Operation/mining_operations.db

File size: 1,916,928 bytes

Database validation successful.
SQLite version: 3.45.1
Integrity check: ok


## 2. Database Structure

### Available tables

In [3]:
def quote_identifier(name):
    # Safely quote a table or column identifier
    return '"' + name.replace('"', '""') + '"'

tables_df = pd.read_sql_query("""
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
""", conn)

table_names = tables_df["table_name"].tolist()

print("Available database tables:")
display(tables_df)
print("Total number of tables:", len(tables_df))

Available database tables:


,table_name
0,daily_costs
1,daily_production
2,equipment_performance
3,mine_sites
4,safety_incidents
5,safety_metrics_summary


Total number of tables: 6


### Record count per table

In [4]:
row_counts = []

for table_name in table_names:
    total_rows = pd.read_sql_query(
        f"SELECT COUNT(*) AS total_rows FROM {quote_identifier(table_name)}", conn
    ).iloc[0, 0]

    row_counts.append({"table_name": table_name, "total_rows": int(total_rows)})

row_counts_df = pd.DataFrame(row_counts)
display(row_counts_df)

print("Total rows across all tables:", f"{row_counts_df['total_rows'].sum():,}")

,table_name,total_rows
0,daily_costs,1825
1,daily_production,1825
2,equipment_performance,8030
3,mine_sites,5
4,safety_incidents,360
5,safety_metrics_summary,4


Total rows across all tables: 12,049


**Interpretasi jumlah record**

- `daily_production` dan `daily_costs` masing-masing 1.825 baris, yaitu 5 lokasi × 365 hari.
- `equipment_performance` 8.030 baris, yaitu 22 unit peralatan × 365 hari (dicek pada Section 3).
- `safety_incidents` 360 baris, `mine_sites` 5 baris, dan `safety_metrics_summary` 4 baris.

### Schema summary

In [5]:
schema_results = []

for table_name in table_names:
    schema = pd.read_sql_query(
        f"PRAGMA table_info({quote_identifier(table_name)})", conn
    )
    schema.insert(0, "table_name", table_name)
    schema_results.append(schema)

schema_df = pd.concat(schema_results, ignore_index=True)
schema_df["column_with_type"] = schema_df["name"] + " (" + schema_df["type"] + ")"

schema_summary = (
    schema_df.groupby("table_name")
    .agg(
        columns=("name", "count"),
        primary_key_columns=("pk", "sum"),
        not_null_columns=("notnull", "sum"),
        column_list=("column_with_type", ", ".join),
    )
    .reset_index()
)

display(schema_summary)

,table_name,columns,primary_key_columns,not_null_columns,column_list
0,daily_costs,14,0,0,"date (TEXT), site_id (TEXT), site_name (TEXT), commodity (TEXT), labor_cost_aud (REAL), equipment_cost_aud (REAL), f..."
1,daily_production,16,0,0,"date (TEXT), site_id (TEXT), site_name (TEXT), commodity (TEXT), status (TEXT), tonnes_mined (INTEGER), tonnes_proce..."
2,equipment_performance,16,0,0,"date (TEXT), site_id (TEXT), equipment_id (TEXT), equipment_type (TEXT), model (TEXT), autonomous (INTEGER), availab..."
3,mine_sites,16,0,0,"site_id (TEXT), site_name (TEXT), commodity (TEXT), region (TEXT), latitude (REAL), longitude (REAL), mine_type (TEX..."
4,safety_incidents,11,0,0,"incident_id (TEXT), date (TEXT), site_id (TEXT), site_name (TEXT), incident_type (TEXT), severity (TEXT), descriptio..."
5,safety_metrics_summary,7,0,0,"site_id (TEXT), site_name (TEXT), total_hours_worked (INTEGER), lti_count (INTEGER), mti_count (INTEGER), trifr (REA..."


**Interpretasi schema**

| Tabel | Kolom | Isi utama |
|---|---|---|
| `daily_production` | 16 | Tonase ditambang dan diproses, grade, recovery, logam, shift, jam operasi, downtime |
| `daily_costs` | 14 | Tujuh komponen biaya, total biaya, `tonnes_produced`, `cost_per_unit` |
| `equipment_performance` | 16 | Jam operasi, availability, utilization, OEE, bahan bakar, biaya perawatan per unit per hari |
| `safety_incidents` | 11 | Jenis, tingkat keparahan, hari hilang, penyebab, status investigasi |
| `safety_metrics_summary` | 7 | Jam kerja, LTI, MTI, TRIFR, LTIFR |
| `mine_sites` | 16 | Data master lokasi dan target |

Struktur ini cocok untuk analisis operasional pertambangan. Definisi metrik dan kualitas nilainya masih perlu dipastikan pada section berikutnya.

### Constraints and indexes

In [6]:
index_df = pd.read_sql_query("""
    SELECT tbl_name AS table_name, name AS index_name
    FROM sqlite_master
    WHERE type = 'index'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY tbl_name, name
""", conn)

display(index_df)

foreign_key_count = sum(
    len(pd.read_sql_query(f"PRAGMA foreign_key_list({quote_identifier(t)})", conn))
    for t in table_names
)

print("Primary-key columns declared:", int(schema_df["pk"].sum()))
print("NOT NULL constraints declared:", int(schema_df["notnull"].sum()))
print("Foreign keys declared:", foreign_key_count)

,table_name,index_name
0,daily_costs,idx_costs_site_date
1,daily_production,idx_production_date
2,daily_production,idx_production_site_date
3,equipment_performance,idx_equipment_id
4,equipment_performance,idx_equipment_site_date
5,safety_incidents,idx_safety_site_date


Primary-key columns declared: 0
NOT NULL constraints declared: 0
Foreign keys declared: 0


**Interpretasi constraint**

- Tidak ada primary key, constraint `NOT NULL`, maupun foreign key yang dideklarasikan di tabel mana pun. Hanya ada 6 index (pada site/tanggal dan `equipment_id`), dan `PRAGMA integrity_check` menyatakan database utuh.
- Database tidak menjamin keunikan key atau keterkaitan antar tabel. Karena itu keduanya diperiksa manual pada Section 5, 6, dan 9.

### Data preview

In [7]:
for table_name in tables_df["table_name"]:
    quoted_name = '"' + table_name.replace('"', '""') + '"'

    print(f"\nTable: {table_name}")

    sample_df = pd.read_sql_query(
        f"SELECT * FROM {quoted_name} LIMIT 5",
        conn
    )

    display(sample_df)


Table: daily_costs


,date,site_id,site_name,commodity,labor_cost_aud,equipment_cost_aud,fuel_cost_aud,maintenance_cost_aud,consumables_cost_aud,utilities_cost_aud,admin_cost_aud,total_cost_aud,tonnes_produced,cost_per_unit
0,2025-01-01,IRON001,Iron Ridge Mine,Iron Ore,1276339.67,893437.77,574352.85,255267.93,127633.97,31908.49,31908.49,3190849.18,113905,28.01
1,2025-01-02,IRON001,Iron Ridge Mine,Iron Ore,1409567.89,986697.52,634305.55,281913.58,140956.79,35239.20,35239.20,3523919.72,127238,27.70
2,2025-01-03,IRON001,Iron Ridge Mine,Iron Ore,1315791.93,921054.35,592106.37,263158.39,131579.19,32894.80,32894.80,3289479.83,124194,26.49
3,2025-01-04,IRON001,Iron Ridge Mine,Iron Ore,1297052.15,907936.50,583673.47,259410.43,129705.21,32426.30,32426.30,3242630.37,117527,27.59
4,2025-01-05,IRON001,Iron Ridge Mine,Iron Ore,1506399.44,1054479.61,677879.75,301279.89,150639.94,37659.99,37659.99,3765998.60,143865,26.18



Table: daily_production


,date,site_id,site_name,commodity,status,tonnes_mined,tonnes_processed,head_grade,recovery_rate,metal_produced,plan_adherence_pct,shift_1_tonnes,shift_2_tonnes,operational_hours,downtime_hours,weather_delay_hours
0,2025-01-01,IRON001,Iron Ridge Mine,Iron Ore,Operational,113905,111626,59.26,99.2,110702.98,83.1,58104,55801,24.0,0.0,0.0
1,2025-01-02,IRON001,Iron Ridge Mine,Iron Ore,Operational,127238,124693,64.91,99.0,123414.60,92.9,59958,67280,24.0,0.0,0.0
2,2025-01-03,IRON001,Iron Ridge Mine,Iron Ore,Operational,124194,121710,61.58,95.8,116588.96,90.7,63486,60708,24.0,0.0,0.0
3,2025-01-04,IRON001,Iron Ridge Mine,Iron Ore,Operational,117527,115176,61.17,96.8,111446.25,85.8,62115,55412,24.0,0.0,0.0
4,2025-01-05,IRON001,Iron Ridge Mine,Iron Ore,Operational,143865,140987,62.67,95.1,134054.22,105.0,65675,78190,24.0,0.0,0.0



Table: equipment_performance


,date,site_id,equipment_id,equipment_type,model,autonomous,available_hours,operating_hours,idle_hours,maintenance_hours,breakdown_hours,availability_pct,utilization_pct,oee_pct,fuel_consumed_liters,maintenance_cost_aud
0,2025-01-01,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,24.0,0.0,0.0,0.0,100.0,100.0,100.0,4320,1537.80
1,2025-01-02,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,20.7,3.3,0.0,0.0,100.0,86.1,86.1,3718,1803.65
2,2025-01-03,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,21.1,2.9,0.0,0.0,100.0,88.1,88.1,3805,1197.03
3,2025-01-04,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,22.6,1.4,0.0,0.0,100.0,94.0,94.0,4062,1421.81
4,2025-01-05,IRON001,TRK-IR-001,Haul Truck,CAT 793F AHS,1,24.0,20.4,3.6,0.0,0.0,100.0,85.1,85.1,3677,520.46


... (2 outputs left out)


Table: safety_incidents


,incident_id,date,site_id,site_name,incident_type,severity,description,days_lost,root_cause,investigation_status,workforce_size
0,INC-00001,2025-02-27,IRON001,Iron Ridge Mine,Lost Time Injury (LTI),High,Fall from height,28,Procedural Gap,Closed,850
1,INC-00002,2025-11-24,IRON001,Iron Ridge Mine,Lost Time Injury (LTI),High,Crush injury,9,Human Error,Closed,850
2,INC-00003,2025-01-16,IRON001,Iron Ridge Mine,Medical Treatment Injury (MTI),Medium,Laceration,0,Training Gap,Closed,850
3,INC-00004,2025-01-17,IRON001,Iron Ridge Mine,Medical Treatment Injury (MTI),Medium,Contusion,0,Equipment Failure,Under Investigation,850
4,INC-00005,2025-02-17,IRON001,Iron Ridge Mine,Medical Treatment Injury (MTI),Medium,Sprain,0,Human Error,Closed,850



Table: safety_metrics_summary


,site_id,site_name,total_hours_worked,lti_count,mti_count,trifr,ltifr
0,IRON001,Iron Ridge Mine,3723000,2,8,2.69,0.54
1,GOLD001,Golden Valley Mine,5256000,2,8,1.90,0.38
2,LITH001,Lithium Creek Mine,1971000,2,8,5.07,1.01
3,COPP001,Copper Hills Mine,1226400,2,8,8.15,1.63


## 3. Date and Site Coverage

### Date coverage

In [8]:
date_audit_df = pd.read_sql_query("""
    SELECT 'daily_production' AS table_name, MIN(date) AS first_date, MAX(date) AS last_date,
           COUNT(DISTINCT date) AS distinct_dates,
           SUM(date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]') AS malformed_dates
    FROM daily_production

    UNION ALL

    SELECT 'daily_costs', MIN(date), MAX(date), COUNT(DISTINCT date),
           SUM(date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]')
    FROM daily_costs

    UNION ALL

    SELECT 'equipment_performance', MIN(date), MAX(date), COUNT(DISTINCT date),
           SUM(date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]')
    FROM equipment_performance

    UNION ALL

    SELECT 'safety_incidents', MIN(date), MAX(date), COUNT(DISTINCT date),
           SUM(date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]')
    FROM safety_incidents
""", conn)

display(date_audit_df)

equipment_coverage = pd.read_sql_query("""
    SELECT COUNT(*) AS equipment_units,
           MIN(days) AS min_days_per_unit,
           MAX(days) AS max_days_per_unit
    FROM (
        SELECT equipment_id, COUNT(DISTINCT date) AS days
        FROM equipment_performance
        GROUP BY equipment_id
    )
""", conn)

display(equipment_coverage)

,table_name,first_date,last_date,distinct_dates,malformed_dates
0,daily_production,2025-01-01,2025-12-31,365,0
1,daily_costs,2025-01-01,2025-12-31,365,0
2,equipment_performance,2025-01-01,2025-12-31,365,0
3,safety_incidents,2025-01-01,2025-12-29,230,0


,equipment_units,min_days_per_unit,max_days_per_unit
0,22,365,365


**Interpretasi date coverage**

- `daily_production`, `daily_costs`, dan `equipment_performance` mencakup 365 tanggal (1 Januari–31 Desember 2025). Format tanggal seragam (`YYYY-MM-DD`) di keempat tabel.
- `equipment_performance` terdiri dari 22 unit dengan 365 hari data masing-masing, sehingga tidak ada hari yang hilang per unit.
- `safety_incidents` hanya memiliki 230 tanggal berbeda (1 Januari–29 Desember). Itu wajar untuk data kejadian, karena tidak setiap hari ada insiden, sehingga bukan indikasi data hilang.

### Site coverage across tables

In [9]:
site_tables = [
    "mine_sites", "daily_production", "daily_costs",
    "equipment_performance", "safety_incidents", "safety_metrics_summary",
]

coverage = {}
for table_name in site_tables:
    coverage[table_name] = (
        pd.read_sql_query(
            f"SELECT site_id, COUNT(*) AS n FROM {quote_identifier(table_name)} GROUP BY site_id",
            conn
        ).set_index("site_id")["n"]
    )

site_coverage = pd.DataFrame(coverage).fillna(0).astype(int)
display(site_coverage)

orphan_sites = site_coverage[site_coverage["mine_sites"] == 0]
print("Site IDs found in other tables but missing from mine_sites:", len(orphan_sites))

,mine_sites,daily_production,daily_costs,equipment_performance,safety_incidents,safety_metrics_summary
site_id,,,,,,
COPP001,1,365,365,1825,90,1
GOLD001,1,365,365,2555,90,1
IRON001,1,365,365,2190,90,1
LITH001,1,365,365,1460,90,1
NICK001,1,365,365,0,0,0


Site IDs found in other tables but missing from mine_sites: 0


**Interpretasi cakupan lokasi**

- Kelima `site_id` di `mine_sites` muncul di `daily_production` dan `daily_costs`, masing-masing 365 baris. Tidak ada `site_id` yatim (yang tidak ada di `mine_sites`).
- Nickel Point (`NICK001`) tidak ada di `equipment_performance`, `safety_incidents`, dan `safety_metrics_summary`. Lokasi ini berstatus *Care & Maintenance*, sehingga ketiadaannya masuk akal. Analisis peralatan dan keselamatan hanya mencakup empat lokasi.

## 4. Missing Values

In [10]:
missing_results = []

for table_name in tables_df["table_name"]:
    quoted_name = '"' + table_name.replace('"', '""') + '"'
    df = pd.read_sql_query(
        f"SELECT * FROM {quoted_name}",
        conn
    )

    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        missing_results.append({
            "table_name": table_name,
            "column_name": column,
            "missing_count": missing_count,
            "missing_percentage": round(
                df[column].isna().mean() * 100, 2
            )
        })

missing_df = pd.DataFrame(missing_results)

missing_df = missing_df.sort_values(
    ["missing_count", "table_name"],
    ascending=[False, True]
)

print("Columns containing missing values:")
display(missing_df[missing_df["missing_count"] > 0])

print("Missing-value check completed.")

Columns containing missing values:


,table_name,column_name,missing_count,missing_percentage
59,mine_sites,automation_level,1,20.0


Missing-value check completed.


In [11]:
missing_site_df = pd.read_sql_query("""
    SELECT site_id, site_name, status, workforce_count, automation_level
    FROM mine_sites
    WHERE automation_level IS NULL
""", conn)

print("Mine sites with missing automation_level:")
display(missing_site_df)

Mine sites with missing automation_level:


,site_id,site_name,status,workforce_count,automation_level
0,NICK001,Nickel Point Mine,Care & Maintenance,25,None


**Interpretasi nilai kosong**

- Hanya ada satu nilai kosong di seluruh database: `mine_sites.automation_level` untuk Nickel Point Mine (`NICK001`), yang berstatus *Care & Maintenance* dengan 25 tenaga kerja.
- Nilai ini lebih tepat dibaca sebagai "tidak berlaku" untuk lokasi yang tidak berproduksi. Nilai tersebut tidak diisi atau dihapus.

## 5. Duplicate Rows and Key Uniqueness

Dua pemeriksaan: baris yang identik seluruhnya, dan duplikasi pada kolom yang seharusnya menjadi key. Karena database tidak mendeklarasikan primary key (Section 2), pemeriksaan key dilakukan manual.

In [12]:
duplicate_results = []

for table_name in table_names:
    df = pd.read_sql_query(f"SELECT * FROM {quote_identifier(table_name)}", conn)
    duplicate_results.append({
        "table_name": table_name,
        "exact_duplicate_rows": int(df.duplicated().sum()),
    })

duplicates_df = pd.DataFrame(duplicate_results)

# Expected key of each table
key_definitions = {
    "mine_sites": "site_id",
    "safety_metrics_summary": "site_id",
    "daily_production": "date, site_id",
    "daily_costs": "date, site_id",
    "equipment_performance": "date, equipment_id",
    "safety_incidents": "incident_id",
}

key_results = []
for table_name, key_columns in key_definitions.items():
    duplicate_keys = pd.read_sql_query(f"""
        SELECT COUNT(*) AS n
        FROM (
            SELECT {key_columns}
            FROM {quote_identifier(table_name)}
            GROUP BY {key_columns}
            HAVING COUNT(*) > 1
        )
    """, conn).iloc[0, 0]

    key_results.append({
        "table_name": table_name,
        "key_columns": key_columns,
        "duplicate_keys": int(duplicate_keys),
    })

uniqueness_df = duplicates_df.merge(pd.DataFrame(key_results), on="table_name")
display(uniqueness_df)

,table_name,exact_duplicate_rows,key_columns,duplicate_keys
0,daily_costs,0,"date, site_id",0
1,daily_production,0,"date, site_id",0
2,equipment_performance,0,"date, equipment_id",0
3,mine_sites,0,site_id,0
4,safety_incidents,0,incident_id,0
5,safety_metrics_summary,0,site_id,0


**Interpretasi duplikasi**

Tidak ada baris identik dan tidak ada key ganda di keenam tabel: `(date, site_id)` unik pada produksi dan biaya, `(date, equipment_id)` unik pada peralatan, dan `incident_id` unik pada insiden. Satu baris mewakili satu lokasi per hari (atau satu unit per hari), sehingga tabel aman digunakan tanpa deduplikasi.

## 6. Production and Cost Reconciliation

**Objective:** Verify that daily production records and daily cost records match one-to-one using `date` and `site_id` as the composite key.

**Checks:** record counts, production records without matching cost, and cost records without matching production.

In [13]:
production_keys = pd.read_sql_query("SELECT date, site_id FROM daily_production", conn)
cost_keys = pd.read_sql_query("SELECT date, site_id FROM daily_costs", conn)

# Standardize key columns before comparison
for df in [production_keys, cost_keys]:
    df["date"] = df["date"].astype(str).str.strip()
    df["site_id"] = df["site_id"].astype(str).str.strip()

production_key_set = set(zip(production_keys["date"], production_keys["site_id"]))
cost_key_set = set(zip(cost_keys["date"], cost_keys["site_id"]))

production_without_cost = production_key_set - cost_key_set
cost_without_production = cost_key_set - production_key_set

print("Production records:", len(production_keys))
print("Cost records:", len(cost_keys))
print("Production keys without matching cost:", len(production_without_cost))
print("Cost keys without matching production:", len(cost_without_production))

Production records: 1825
Cost records: 1825
Production keys without matching cost: 0
Cost keys without matching production: 0


**Interpretasi rekonsiliasi**

Semua 1.825 record produksi memiliki pasangan di tabel biaya, dan sebaliknya. Kedua tabel dapat digabung (*join*) dengan `date` dan `site_id` tanpa kehilangan baris.

## 7. Cost Consistency

Memeriksa apakah `total_cost_aud` sama dengan jumlah tujuh komponen biaya. Nilai kosong pada komponen diperlakukan sebagai nol hanya untuk pemeriksaan ini, bukan sebagai perbaikan data.

In [14]:
components = [
    "labor_cost_aud", "equipment_cost_aud", "fuel_cost_aud", "maintenance_cost_aud",
    "consumables_cost_aud", "utilities_cost_aud", "admin_cost_aud",
]

cost_check_df = pd.read_sql_query("SELECT date, site_id, " + ", ".join(components) + ", total_cost_aud FROM daily_costs", conn)

cost_check_df["calculated_total_cost"] = cost_check_df[components].fillna(0).sum(axis=1)
cost_check_df["cost_difference"] = cost_check_df["total_cost_aud"] - cost_check_df["calculated_total_cost"]

# Round to cents first: comparing raw floating-point values against 0.01 is unstable
cost_check_df["cost_difference_rounded"] = cost_check_df["cost_difference"].round(2)

records = len(cost_check_df)
nonzero = int((cost_check_df["cost_difference_rounded"] != 0).sum())

print("Records checked:", records)
print("Raw differences above AUD 0.01 (unstable test):", int((cost_check_df["cost_difference"].abs() > 0.01).sum()))
print("Records with a non-zero difference after rounding to cents:", nonzero, f"({nonzero / records:.1%})")
print("Largest absolute difference (AUD):", cost_check_df["cost_difference_rounded"].abs().max())

difference_summary = (
    cost_check_df["cost_difference_rounded"]
    .value_counts()
    .sort_index()
    .rename_axis("difference_aud")
    .reset_index(name="record_count")
)
display(difference_summary)

# Where do the differences occur?
display(
    cost_check_df.assign(has_difference=cost_check_df["cost_difference_rounded"] != 0)
    .groupby("site_id")["has_difference"]
    .agg(records="count", records_with_difference="sum")
    .reset_index()
)

Records checked: 1825
Raw differences above AUD 0.01 (unstable test): 506
Records with a non-zero difference after rounding to cents: 842 (46.1%)
Largest absolute difference (AUD): 0.02


,difference_aud,record_count
0,-0.02,56
1,-0.01,374
2,0.00,983
3,0.01,379
4,0.02,33


,site_id,records,records_with_difference
0,COPP001,365,210
1,GOLD001,365,211
2,IRON001,365,216
3,LITH001,365,205
4,NICK001,365,0


### Cost structure: share of each component in `total_cost_aud`

In [15]:
share_df = cost_check_df[components].div(cost_check_df["total_cost_aud"], axis=0)
share_df["site_id"] = cost_check_df["site_id"]

share_mean = share_df.groupby("site_id")[components].mean().round(4)
share_spread = share_df.groupby("site_id")[components].agg(lambda s: s.max() - s.min())

display(share_mean)
print("Largest spread of any component share within a site:", round(share_spread.to_numpy().max(), 6))

,labor_cost_aud,equipment_cost_aud,fuel_cost_aud,maintenance_cost_aud,consumables_cost_aud,utilities_cost_aud,admin_cost_aud
site_id,,,,,,,
COPP001,0.4000,0.28,0.18,0.0800,0.04,0.0100,0.0100
GOLD001,0.4000,0.28,0.18,0.0800,0.04,0.0100,0.0100
IRON001,0.4000,0.28,0.18,0.0800,0.04,0.0100,0.0100
LITH001,0.4000,0.28,0.18,0.0800,0.04,0.0100,0.0100
NICK001,0.5882,0.00,0.00,0.2353,0.00,0.1176,0.0588


Largest spread of any component share within a site: 0.0


**Interpretasi konsistensi biaya**

| Selisih (AUD) | Jumlah record |
|---|---|
| 0,00 | 983 |
| ±0,01 | 753 |
| ±0,02 | 89 |

- Dari 1.825 record, 983 sama persis dan 842 (46,1%) berbeda satu atau dua sen. Tidak ada selisih lebih besar dari AUD 0,02. Uji langsung pada angka mentah dengan batas 0,01 tidak stabil (menandai 506 record), sehingga selisih dibulatkan ke sen terlebih dahulu.
- Selisih hanya terjadi di empat lokasi yang berproduksi. Nickel Point tidak memiliki selisih sama sekali.
- Pada empat lokasi tersebut setiap komponen biaya adalah **porsi tetap** dari total: labor 40%, equipment 28%, fuel 18%, maintenance 8%, consumables 4%, utilities 1%, dan admin 1%. Penyimpangan porsi antar hari praktis nol. Nickel Point memiliki struktur biaya tetap tersendiri (labor 5.000, maintenance 2.000, utilities 1.000, admin 500 per hari).
- Pola ini konsisten dengan penjelasan bahwa tiap komponen dibulatkan ke sen secara terpisah dari total. Ini penjelasan yang paling masuk akal dari bukti di atas, bukan sesuatu yang dibuktikan dari sumber data.
- Konsekuensi untuk EDA: komposisi biaya (mis. labor 40%) mencerminkan asumsi tetap pada data sintetis, bukan struktur biaya tambang nyata.

## 8. Cost per Unit and Zero-Production Records

Memeriksa tiga hal: (1) apa arti `tonnes_produced` terhadap tabel produksi, (2) lokasi dan record dengan produksi nol, dan (3) pembagi apa yang menghasilkan `cost_per_unit` di setiap lokasi.

In [16]:
unit_audit = pd.read_sql_query("""
    SELECT
        c.date,
        c.site_id,
        c.tonnes_produced,
        c.total_cost_aud,
        c.cost_per_unit,
        p.tonnes_mined,
        p.tonnes_processed,
        p.metal_produced
    FROM daily_costs AS c
    INNER JOIN daily_production AS p
        ON c.date = p.date AND c.site_id = p.site_id
""", conn)

print("Records:", len(unit_audit))
print("tonnes_produced equals tonnes_mined:", int((unit_audit["tonnes_produced"] == unit_audit["tonnes_mined"]).sum()))
print("tonnes_produced equals tonnes_processed:", int((unit_audit["tonnes_produced"] == unit_audit["tonnes_processed"]).sum()))
print("Missing tonnes_produced:", int(unit_audit["tonnes_produced"].isna().sum()))

# Zero-production records by site
zero_production = (
    unit_audit.assign(zero_production=unit_audit["tonnes_produced"] == 0)
    .groupby("site_id")
    .agg(
        records=("date", "count"),
        zero_production_records=("zero_production", "sum"),
        avg_total_cost_aud=("total_cost_aud", "mean"),
        avg_cost_per_unit=("cost_per_unit", "mean"),
    )
    .round(2)
    .reset_index()
)
display(zero_production)

Records: 1825
tonnes_produced equals tonnes_mined: 1825
tonnes_produced equals tonnes_processed: 365
Missing tonnes_produced: 0


,site_id,records,zero_production_records,avg_total_cost_aud,avg_cost_per_unit
0,COPP001,365,0,206664.10,78.76
1,GOLD001,365,0,2660155.66,2355.21
2,IRON001,365,0,3478427.24,27.05
3,LITH001,365,0,1143625.40,72323.40
4,NICK001,365,365,8500.00,0.00


In [17]:
# Which denominator reproduces the stored cost_per_unit? (positive-production records only)
positive = unit_audit[unit_audit["tonnes_produced"] > 0].copy()

candidates = {
    "per_tonne_produced": positive["total_cost_aud"] / positive["tonnes_produced"],
    "per_metal_unit": positive["total_cost_aud"] / positive["metal_produced"],
    "per_metal_unit_x2200": positive["total_cost_aud"] / (positive["metal_produced"] * 2200),
}

for name, calculated in candidates.items():
    positive[f"max_abs_diff_{name}"] = (positive["cost_per_unit"] - calculated).abs()

diff_columns = [f"max_abs_diff_{name}" for name in candidates]

unit_basis = positive.groupby("site_id")[diff_columns].max().round(4)
unit_basis["best_match"] = unit_basis.idxmin(axis=1).str.replace("max_abs_diff_", "", regex=False)

display(unit_basis.reset_index())

,site_id,max_abs_diff_per_tonne_produced,max_abs_diff_per_metal_unit,max_abs_diff_per_metal_unit_x2200,best_match
0,COPP001,4194.1629,193831.6307,0.0050,per_metal_unit_x2200
1,GOLD001,2555.4528,0.0050,2638.0603,per_metal_unit
2,IRON001,0.0050,2.3602,28.3361,per_tonne_produced
3,LITH001,79712.2729,0.0052,80345.6526,per_metal_unit


**Interpretasi cost per unit**

- `tonnes_produced` di `daily_costs` identik dengan `tonnes_mined` di `daily_production` pada seluruh 1.825 record. Kecocokan dengan `tonnes_processed` hanya terjadi pada 365 record Nickel Point yang sama-sama bernilai nol. Meskipun bernama "produced", kolom ini adalah tonase yang ditambang.
- Ada 365 record dengan produksi nol, seluruhnya milik Nickel Point (`NICK001`), yang tetap mencatat biaya A$8.500 per hari. `cost_per_unit` pada record ini diisi 0, padahal biaya per unit tidak terdefinisi ketika produksi nol. Untuk analisis, nilai ini diperlakukan sebagai tidak terdefinisi (`NaN`), bukan 0.
- Pembagi `cost_per_unit` **berbeda antar lokasi**:
  - Iron Ridge: biaya ÷ `tonnes_produced` (selisih maksimum sekitar A$0,005).
  - Golden Valley dan Lithium Creek: biaya ÷ `metal_produced`.
  - Copper Hills: biaya ÷ (`metal_produced` × sekitar 2.200), mendekati jumlah pon per ton (2.204,6), sehingga angkanya per pon.
- Audit sebelumnya hanya melihat 10 baris pertama (Iron Ridge) dan menyimpulkan nilai konsisten dengan biaya ÷ tonase. Itu hanya berlaku untuk Iron Ridge. `cost_per_unit` tidak boleh dibandingkan langsung antar lokasi.
- Satuan logam tidak didokumentasikan di dataset. Penjelasan di atas adalah kesimpulan dari pola angka, bukan dari dokumentasi sumber.

## 9. Value Range and Logical Consistency Checks

Pemeriksaan berbasis aturan pada nilai dan hubungan antar tabel. Setiap aturan dihitung sebagai jumlah baris yang melanggar, dan hasil yang diharapkan adalah 0. Toleransi 0,15 jam dan 1,0 poin persen dipakai karena jam peralatan dicatat dengan ketelitian 0,1 jam.

In [18]:
rules = [
    # (table, rule, SQL returning the number of violating rows)
    ("daily_production", "tonnes_processed greater than tonnes_mined",
     "SELECT COUNT(*) FROM daily_production WHERE tonnes_processed > tonnes_mined"),
    ("daily_production", "negative tonnes or metal",
     "SELECT COUNT(*) FROM daily_production WHERE tonnes_mined < 0 OR tonnes_processed < 0 OR metal_produced < 0"),
    ("daily_production", "shift_1 + shift_2 differs from tonnes_mined",
     "SELECT COUNT(*) FROM daily_production WHERE shift_1_tonnes + shift_2_tonnes != tonnes_mined"),
    ("daily_production", "recovery_rate outside 0-100",
     "SELECT COUNT(*) FROM daily_production WHERE recovery_rate < 0 OR recovery_rate > 100"),
    ("daily_production", "operational_hours outside 0-24",
     "SELECT COUNT(*) FROM daily_production WHERE operational_hours < 0 OR operational_hours > 24"),
    ("daily_production", "operational + downtime + weather hours differ from 24",
     "SELECT COUNT(*) FROM daily_production WHERE ABS(operational_hours + downtime_hours + weather_delay_hours - 24) > 0.01"),
    ("daily_production", "status differs from mine_sites",
     "SELECT COUNT(*) FROM daily_production p JOIN mine_sites m USING (site_id) WHERE p.status != m.status"),
    ("daily_production", "site_name or commodity differs from mine_sites",
     "SELECT COUNT(*) FROM daily_production p JOIN mine_sites m USING (site_id) WHERE p.site_name != m.site_name OR p.commodity != m.commodity"),
    ("daily_costs", "negative cost value",
     "SELECT COUNT(*) FROM daily_costs WHERE labor_cost_aud < 0 OR equipment_cost_aud < 0 OR fuel_cost_aud < 0 OR maintenance_cost_aud < 0 OR consumables_cost_aud < 0 OR utilities_cost_aud < 0 OR admin_cost_aud < 0 OR total_cost_aud < 0"),
    ("daily_costs", "site_name or commodity differs from mine_sites",
     "SELECT COUNT(*) FROM daily_costs c JOIN mine_sites m USING (site_id) WHERE c.site_name != m.site_name OR c.commodity != m.commodity"),
    ("equipment_performance", "availability, utilization, or OEE outside 0-100",
     "SELECT COUNT(*) FROM equipment_performance WHERE availability_pct NOT BETWEEN 0 AND 100 OR utilization_pct NOT BETWEEN 0 AND 100 OR oee_pct NOT BETWEEN 0 AND 100"),
    ("equipment_performance", "negative hours, fuel, or cost",
     "SELECT COUNT(*) FROM equipment_performance WHERE operating_hours < 0 OR idle_hours < 0 OR maintenance_hours < 0 OR breakdown_hours < 0 OR fuel_consumed_liters < 0 OR maintenance_cost_aud < 0"),
    ("equipment_performance", "operating + idle hours differ from available hours",
     "SELECT COUNT(*) FROM equipment_performance WHERE ABS(operating_hours + idle_hours - available_hours) > 0.15"),
    ("equipment_performance", "available hours differ from 24 - maintenance - breakdown",
     "SELECT COUNT(*) FROM equipment_performance WHERE ABS(available_hours - (24 - maintenance_hours - breakdown_hours)) > 0.15"),
    ("equipment_performance", "availability_pct differs from available_hours / 24",
     "SELECT COUNT(*) FROM equipment_performance WHERE ABS(availability_pct - available_hours * 100.0 / 24) > 1.0"),
    ("equipment_performance", "utilization_pct differs from operating / available hours",
     "SELECT COUNT(*) FROM equipment_performance WHERE ABS(utilization_pct - operating_hours * 100.0 / available_hours) > 1.0"),
    ("equipment_performance", "OEE differs from availability x utilization",
     "SELECT COUNT(*) FROM equipment_performance WHERE ABS(oee_pct - availability_pct * utilization_pct / 100.0) > 0.5"),
    ("equipment_performance", "equipment_id with more than one type, model, or mode",
     "SELECT COUNT(*) FROM (SELECT equipment_id FROM equipment_performance GROUP BY equipment_id HAVING COUNT(DISTINCT equipment_type) > 1 OR COUNT(DISTINCT model) > 1 OR COUNT(DISTINCT autonomous) > 1)"),
    ("safety_incidents", "date outside 2025",
     "SELECT COUNT(*) FROM safety_incidents WHERE date < '2025-01-01' OR date > '2025-12-31'"),
    ("safety_incidents", "negative days_lost",
     "SELECT COUNT(*) FROM safety_incidents WHERE days_lost < 0"),
    ("safety_incidents", "days_lost above 0 on a non-LTI incident",
     "SELECT COUNT(*) FROM safety_incidents WHERE days_lost > 0 AND incident_type NOT LIKE '%(LTI)%'"),
    ("safety_incidents", "LTI with zero days_lost",
     "SELECT COUNT(*) FROM safety_incidents WHERE incident_type LIKE '%(LTI)%' AND days_lost = 0"),
    ("safety_incidents", "workforce_size differs from mine_sites.workforce_count",
     "SELECT COUNT(*) FROM safety_incidents s JOIN mine_sites m USING (site_id) WHERE s.workforce_size != m.workforce_count"),
    ("safety_metrics_summary", "TRIFR differs from (LTI + MTI) x 1e6 / hours",
     "SELECT COUNT(*) FROM safety_metrics_summary WHERE ABS(trifr - (lti_count + mti_count) * 1e6 / total_hours_worked) > 0.01"),
    ("safety_metrics_summary", "LTIFR differs from LTI x 1e6 / hours",
     "SELECT COUNT(*) FROM safety_metrics_summary WHERE ABS(ltifr - lti_count * 1e6 / total_hours_worked) > 0.01"),
    ("safety_metrics_summary", "LTI or MTI count differs from safety_incidents",
     "SELECT COUNT(*) FROM safety_metrics_summary s WHERE lti_count != (SELECT COUNT(*) FROM safety_incidents i WHERE i.site_id = s.site_id AND i.incident_type LIKE '%(LTI)%') OR mti_count != (SELECT COUNT(*) FROM safety_incidents i WHERE i.site_id = s.site_id AND i.incident_type LIKE '%(MTI)%')"),
    ("mine_sites", "annual target differs from daily target x 365 by more than 2%",
     "SELECT COUNT(*) FROM mine_sites WHERE annual_target_tonnes > 0 AND ABS(annual_target_tonnes - daily_target_tonnes * 365) * 1.0 / annual_target_tonnes > 0.02"),
]

rule_results = []
for table_name, rule, sql in rules:
    violations = int(pd.read_sql_query(sql, conn).iloc[0, 0])
    rule_results.append({
        "table_name": table_name,
        "rule": rule,
        "violating_rows": violations,
        "status": "Pass" if violations == 0 else "Review",
    })

rule_results_df = pd.DataFrame(rule_results)
display(rule_results_df)

print("Rules tested:", len(rule_results_df))
print("Rules passed:", int((rule_results_df["status"] == "Pass").sum()))
print("Rules to review:", int((rule_results_df["status"] == "Review").sum()))

,table_name,rule,violating_rows,status
0,daily_production,tonnes_processed greater than tonnes_mined,0,Pass
1,daily_production,negative tonnes or metal,0,Pass
2,daily_production,shift_1 + shift_2 differs from tonnes_mined,0,Pass
3,daily_production,recovery_rate outside 0-100,0,Pass
4,daily_production,operational_hours outside 0-24,0,Pass
5,daily_production,operational + downtime + weather hours differ from 24,2,Review
6,daily_production,status differs from mine_sites,0,Pass
7,daily_production,site_name or commodity differs from mine_sites,0,Pass
8,daily_costs,negative cost value,0,Pass
9,daily_costs,site_name or commodity differs from mine_sites,0,Pass


Rules tested: 27
Rules passed: 26
Rules to review: 1


In [19]:
# Rule flagged for review: days where the daily hours exceed 24
hours_review = pd.read_sql_query("""
    SELECT date, site_id, operational_hours, downtime_hours, weather_delay_hours,
           ROUND(operational_hours + downtime_hours + weather_delay_hours, 1) AS total_hours
    FROM daily_production
    WHERE ABS(operational_hours + downtime_hours + weather_delay_hours - 24) > 0.01
""", conn)

display(hours_review)

# Informational: days above 100% of plan are plausible and not treated as errors
over_plan = pd.read_sql_query(
    "SELECT COUNT(*) AS days_above_100_pct_plan FROM daily_production WHERE plan_adherence_pct > 100", conn
)
display(over_plan)

,date,site_id,operational_hours,downtime_hours,weather_delay_hours,total_hours
0,2025-01-22,LITH001,12.0,12.0,12.0,36.0
1,2025-03-11,LITH001,16.8,7.2,7.2,31.2


,days_above_100_pct_plan
0,372


**Interpretasi aturan nilai**

- 27 aturan diuji dan **26 lolos**. Nilai tidak ada yang negatif, persentase berada di rentang 0–100, shift 1 + shift 2 selalu sama dengan `tonnes_mined`, dan tonase diproses tidak pernah melebihi tonase ditambang.
- Jam peralatan konsisten: operating + idle = available, available = 24 − maintenance − breakdown, availability dan utilization sesuai definisinya, dan **OEE = availability × utilization** pada seluruh 8.030 baris. Artinya OEE pada dataset ini tidak memiliki komponen performance atau quality (ini dipakai pada analisis EDA).
- Metrik keselamatan konsisten: TRIFR dan LTIFR dapat dihitung ulang dari jam kerja, dan jumlah LTI/MTI sama dengan catatan insiden. Hari hilang hanya muncul pada insiden LTI.
- Satu aturan perlu ditinjau: pada Lithium Creek tanggal 22 Januari dan 11 Maret, jumlah jam operasional + downtime + weather delay melebihi 24 jam (36,0 dan 31,2). Kemungkinan weather delay dicatat tumpang tindih dengan downtime. Dampaknya kecil (dua baris), tetapi jam pada dua hari itu tidak boleh dijumlahkan begitu saja.
- Sebanyak 372 hari memiliki `plan_adherence_pct` di atas 100%. Ini tidak dianggap kesalahan, karena produksi di atas rencana adalah hal yang wajar.

## 10. Audit Summary and Conclusion

In [20]:
tables_with_missing = missing_df.loc[missing_df["missing_count"] > 0, "table_name"].nunique()
tables_with_duplicates = int((uniqueness_df["exact_duplicate_rows"] > 0).sum())
tables_with_duplicate_keys = int((uniqueness_df["duplicate_keys"] > 0).sum())

print("=" * 55)
print("MINING OPERATIONS DATABASE AUDIT SUMMARY")
print("=" * 55)
print("Database file:", DB_PATH)
print("Number of tables:", len(tables_df))
print("Total rows across tables:", f"{row_counts_df['total_rows'].sum():,}")
print("Tables with missing values:", tables_with_missing)
print("Tables with exact duplicate rows:", tables_with_duplicates)
print("Tables with duplicate keys:", tables_with_duplicate_keys)
print("Unmatched production/cost keys:", len(production_without_cost) + len(cost_without_production))
print("Cost records with cent-level differences:", nonzero, f"(max AUD {cost_check_df['cost_difference_rounded'].abs().max()})")
print("Logical rules passed:", f"{int((rule_results_df['status'] == 'Pass').sum())} of {len(rule_results_df)}")

print("\nAudit completed. No original values were modified.")

MINING OPERATIONS DATABASE AUDIT SUMMARY
Database file: /content/drive/MyDrive/Mining-Operation/mining_operations.db
Number of tables: 6
Total rows across tables: 12,049
Tables with missing values: 1
Tables with exact duplicate rows: 0
Tables with duplicate keys: 0
Unmatched production/cost keys: 0
Cost records with cent-level differences: 842 (max AUD 0.02)
Logical rules passed: 26 of 27

Audit completed. No original values were modified.


## Final Audit Conclusion

The SQLite database contains six tables with a total of 12,049 records. It passed SQLite's integrity check, and no original values were modified during the audit.

### Findings

1. **Structure:** Six tables were identified. No primary keys, `NOT NULL` constraints, or foreign keys are declared, so uniqueness and referential integrity were validated manually.
2. **Missing values:** One missing value exists, in `mine_sites.automation_level` for Nickel Point Mine (Care & Maintenance). It was not imputed.
3. **Duplicates:** No exact duplicate rows and no duplicate keys were found in any table.
4. **Coverage:** Production, cost, and equipment data cover all 365 days of 2025. Equipment covers 22 units over 365 days each. Nickel Point has no equipment or safety records, which is consistent with its status.
5. **Production–cost reconciliation:** All 1,825 production records match cost records on `date` and `site_id`.
6. **Cost consistency:** `total_cost_aud` differs from the sum of components by at most AUD 0.02 in 842 records (46.1%), only at producing sites. Components are fixed shares of the total (40/28/18/8/4/1/1%), which is consistent with cent-level rounding.
7. **Unit cost:** `tonnes_produced` equals `tonnes_mined`. `cost_per_unit` is calculated on a different basis per site (per tonne for Iron Ridge, per metal unit for Golden Valley and Lithium Creek, per pound-equivalent for Copper Hills), and is recorded as 0 for 365 zero-production days at Nickel Point. It should not be compared directly across sites.
8. **Logical rules:** 26 of 27 rules passed. One rule flagged two Lithium Creek days where operational, downtime, and weather-delay hours exceed 24. OEE equals availability × utilization.

### Conclusion

The database is structurally suitable for exploratory analysis, subject to the limitations above. The data is synthetic, so the findings demonstrate an audit workflow and do not describe a real mining operation.

The next phase is Exploratory Data Analysis in `02_Mining_Operations_EDA.ipynb`.